# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victorydavid-lab/Victory/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
# FlyRank Week 4 — data connection

%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

# Get Hugging Face token from Colab Secret
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

os.environ["HF_TOKEN"] = HF_TOKEN

# Connect DuckDB
con = duckdb.connect()

# Load HTTPFS for Hugging Face
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
}

print("FlyRank Week 4 connection ready.")
print(TABLES)

FlyRank Week 4 connection ready.
{'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"}


## 1. My rule and its reason codes

Signal 1 — GSC impressions (volume): CONFIRMED. Higher-impression content receives substantially more average clicks in the March 2026 panel. The relationship with average position is less consistent at higher volumes, so impressions are useful as a volume/opportunity signal rather than proof of ranking quality.

Signal 2 — CTR relative to average position: CONFIRMED. CTR is substantially lower for content ranking at position 21+ (0.13%) than for the stronger-ranking buckets (0.29–0.38%). The relationship is not perfectly monotonic across every bucket, so position should be treated as a useful ranking-context signal rather than a complete explanation of CTR.

In [8]:
# Signal check 1: GSC impressions / search volume

query_volume = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions = 0 THEN '0'
        WHEN gsc_impressions < 10 THEN '1-9'
        WHEN gsc_impressions < 50 THEN '10-49'
        WHEN gsc_impressions < 100 THEN '50-99'
        WHEN gsc_impressions < 500 THEN '100-499'
        ELSE '500+'
    END AS impressions_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_clicks), 2) AS avg_clicks,
    ROUND(AVG(gsc_avg_position), 2) AS avg_position
FROM {TABLES['fact_daily']}
WHERE gsc_data_available IS TRUE
GROUP BY 1
ORDER BY
    CASE impressions_bucket
        WHEN '0' THEN 1
        WHEN '1-9' THEN 2
        WHEN '10-49' THEN 3
        WHEN '50-99' THEN 4
        WHEN '100-499' THEN 5
        WHEN '500+' THEN 6
    END
""").df()

query_volume

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impressions_bucket,n,avg_clicks,avg_position
0,1-9,1463532,0.01,20.52
1,10-49,1110087,0.06,14.16
2,50-99,398834,0.22,10.88
3,100-499,537157,0.65,10.94
4,500+,101451,2.94,11.69


Signal 1 — GSC impressions (volume): CONFIRMED. Higher-impression content receives substantially more average clicks in the March 2026 panel. The relationship with average position is less consistent at higher volumes, so impressions are useful as a volume/opportunity signal rather than proof of ranking quality.

In [9]:
# Signal check 2: CTR relative to average position

query_ctr_position = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 5 THEN '4-5'
        WHEN gsc_avg_position <= 10 THEN '6-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,

    COUNT(*) AS n,

    ROUND(
        100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0),
        2
    ) AS ctr_percent,

    ROUND(AVG(gsc_impressions), 2) AS avg_impressions,

    ROUND(AVG(gsc_clicks), 2) AS avg_clicks

FROM {TABLES['fact_daily']}

WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL

GROUP BY 1

ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-5' THEN 2
        WHEN '6-10' THEN 3
        WHEN '11-20' THEN 4
        WHEN '21+' THEN 5
    END
""").df()

query_ctr_position

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,ctr_percent,avg_impressions,avg_clicks
0,1-3,727362,0.38,74.28,0.28
1,4-5,535763,0.36,126.69,0.45
2,6-10,920359,0.29,76.01,0.22
3,11-20,519223,0.31,56.60,0.18
4,21+,908354,0.13,65.41,0.09


Signal 2 — CTR relative to average position: CONFIRMED. CTR is substantially lower for content ranking at position 21+ (0.13%) than for the stronger-ranking buckets (0.29–0.38%). The relationship is not perfectly monotonic across every bucket, so position should be treated as a useful ranking-context signal rather than a complete explanation of CTR.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Baseline rule: Rank content items by combining search opportunity (impressions) with ranking weakness (average position). Higher impressions indicate more search opportunity, while a higher average position number indicates weaker ranking. The highest-scoring items are therefore content items with meaningful search visibility but weaker positions, which are potential opportunities for improvement.

Reason code: HIGH_VOLUME_LOW_POSITION

Action label: REVIEW_SEO_OPPORTUNITY

This is a simple baseline intended to prioritize content for review, not to claim that the rule causes ranking improvement.

In [10]:
# Build the baseline ranked queue for March 2026

baseline_queue = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    CASE
        WHEN gsc_impressions >= 500 THEN 5
        WHEN gsc_impressions >= 100 THEN 4
        WHEN gsc_impressions >= 50 THEN 3
        WHEN gsc_impressions >= 10 THEN 2
        WHEN gsc_impressions >= 1 THEN 1
        ELSE 0
    END AS volume_score,

    CASE
        WHEN gsc_avg_position > 20 THEN 5
        WHEN gsc_avg_position > 10 THEN 4
        WHEN gsc_avg_position > 5 THEN 3
        WHEN gsc_avg_position > 3 THEN 2
        WHEN gsc_avg_position > 0 THEN 1
        ELSE 0
    END AS position_score

FROM {TABLES['fact_daily']}

WHERE gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL
  AND gsc_avg_position IS NOT NULL
""").df()

# Combine the two signals into one simple baseline score
baseline_queue["score"] = (
    baseline_queue["volume_score"]
    + baseline_queue["position_score"]
)

# One reason code and one action label
baseline_queue["reason_code"] = "HIGH_VOLUME_LOW_POSITION"
baseline_queue["action_label"] = "REVIEW_SEO_OPPORTUNITY"

# Rank highest scores first
baseline_queue = baseline_queue.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_queue["rank"] = baseline_queue.index + 1

# Write the ranked queue
import os

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Ranked queue created.")
print(f"Rows ranked: {len(baseline_queue):,}")
print("Saved to: work/outputs/baseline_action_score.csv")

baseline_queue.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue created.
Rows ranked: 3,611,061
Saved to: work/outputs/baseline_action_score.csv


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,volume_score,position_score,score,reason_code,action_label,rank
0,2026-03-31,client_23a62021009f63c4,content_e6df0936699f5b8f,14682,269,25.035826,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,1
1,2026-03-09,client_23a62021009f63c4,content_36e53e9c707674fc,9409,1,32.640344,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,2
2,2026-03-10,client_23a62021009f63c4,content_3df3f32f3fd58dea,9300,14,22.399032,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,3
3,2026-03-11,client_23a62021009f63c4,content_36e53e9c707674fc,9285,16,32.510609,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,4
4,2026-03-12,client_23a62021009f63c4,content_3df3f32f3fd58dea,9274,10,24.985012,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,5
5,2026-03-10,client_23a62021009f63c4,content_36e53e9c707674fc,8963,11,32.476180,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,6
6,2026-03-08,client_23a62021009f63c4,content_36e53e9c707674fc,8842,3,33.982809,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,7
7,2026-03-11,client_23a62021009f63c4,content_3df3f32f3fd58dea,8799,11,23.389476,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,8
8,2026-03-09,client_23a62021009f63c4,content_3df3f32f3fd58dea,8267,10,24.421193,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,9
9,2026-03-12,client_23a62021009f63c4,content_36e53e9c707674fc,8049,12,33.864952,5,5,10,HIGH_VOLUME_LOW_POSITION,REVIEW_SEO_OPPORTUNITY,10


## 3. Top-20 review

## Top-20 review

| Rank | Action | Reason code | Confidence note | What would make it wrong? |
|---:|---|---|---|---|
| 1 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 14,682 impressions and position 25.04 indicate strong visibility but weak ranking. | Wrong if the impressions come from irrelevant or low-value queries, or if the page is not realistically improvable. |
| 2 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 9,409 impressions with position 32.64 shows substantial visibility despite weak ranking. | Wrong if the queries are poorly aligned with the page or the ranking reflects highly competitive search intent. |
| 3 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 9,300 impressions and position 22.40 indicate a clear volume-and-ranking opportunity. | Wrong if improving the page would not attract valuable clicks or the queries are not strategically relevant. |
| 4 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 9,285 impressions and position 32.51 meet both strong-volume and weak-position criteria. | Wrong if the page is targeting queries where a higher ranking is unrealistic. |
| 5 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 9,274 impressions and position 24.99 indicate substantial search visibility with weak ranking. | Wrong if the search demand is low quality or unrelated to the intended audience. |
| 6 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 8,963 impressions and position 32.48 make this a strong baseline priority. | Wrong if the content has little realistic optimisation potential despite its visibility. |
| 7 | REVIEW_SEO_OPPORTUNITY | HIGH — 8,842 impressions and position 33.98 indicate high visibility with particularly weak ranking. | Wrong if the queries are too competitive or poorly matched to the content. |
| 8 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 8,799 impressions and position 23.39 satisfy both scoring conditions. | Wrong if the impressions come from searches that are not valuable to the client. |
| 9 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 8,267 impressions and position 24.42 suggest meaningful search visibility with ranking weakness. | Wrong if the content cannot be improved enough to capture additional demand. |
| 10 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 8,049 impressions and position 33.86 show strong volume but very weak ranking. | Wrong if the ranking reflects an inherently poor query-content fit rather than an optimisation issue. |
| 11 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 8,025 impressions and position 31.18 meet both baseline thresholds. | Wrong if the observed demand is not relevant or actionable for the content. |
| 12 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 7,973 impressions and position 32.58 indicate strong volume and weak ranking. | Wrong if the page has limited potential to rank higher for the underlying queries. |
| 13 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 7,605 impressions and position 33.19 indicate substantial visibility but poor ranking. | Wrong if the queries are highly competitive or not aligned with the page's purpose. |
| 14 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 7,522 impressions and position 30.45 meet both conditions strongly. | Wrong if the impressions do not represent valuable or relevant search demand. |
| 15 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 7,459 impressions and position 33.25 indicate a strong volume/position opportunity. | Wrong if improving the content would not materially change its search performance. |
| 16 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 7,435 impressions and position 31.19 indicate strong visibility but weak ranking. | Wrong if the queries generating impressions are low-intent or poorly matched to the content. |
| 17 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 7,174 impressions and position 32.66 satisfy both scoring thresholds. | Wrong if the page cannot realistically compete for the searches generating that volume. |
| 18 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 7,026 impressions and position 28.10 indicate strong volume with weak ranking. | Wrong if the zero clicks reflect query characteristics rather than an SEO optimisation problem. |
| 19 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 6,887 impressions and position 27.22 indicate substantial visibility with weak ranking. | Wrong if the search demand is not relevant enough to justify optimisation effort. |
| 20 | REVIEW_SEO_OPPORTUNITY | HIGH_VOLUME_LOW_POSITION | High — 6,777 impressions and position 24.50 indicate high volume and weak ranking. | Wrong if the content has little realistic opportunity to improve its position. |

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

## Weak picks + leakage check

### Weak picks

The baseline has a useful but important limitation: it ranks observations at the client-content-date grain rather than unique content items. As a result, the same content item can appear multiple times in the top 20 on different reporting dates. This can cause the queue to repeatedly recommend the same content instead of providing a diverse set of content opportunities.

For example, `content_36e53e9c707674fc` and `content_3df3f32f3fd58dea` appear repeatedly in the top 20. These are therefore weaker picks as separate queue entries, even though the underlying content may genuinely represent an opportunity.

Another potential weakness is that the rule treats high impressions and weak average position as sufficient evidence for an SEO opportunity. A high-impression page may still be a poor recommendation if its queries are irrelevant, highly competitive, or poorly matched to the content's intended search intent.

### Leakage check

No future-window data was used in the baseline score. The rule uses only March 2026 fields available in the March reporting panel: GSC impressions and average position.

No product flags or outcome/label-derived fields were used to calculate the score.

The rule does not use future performance, future rankings, conversions, or any other information that would only become known after the decision moment.

The baseline therefore uses decision-time signals only, although the daily observation grain remains a limitation of the current queue.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.